## LightGBM

The flow is:

1. Train LightGBM on transformed data.
2. Train LightGBM with class weights.
3. Train LightGBM on resampled datasets:
   - ADASYN
   - SMOTE
   - SMOTE-ENN
   - SMOTE-TOMEK
4. Compare all experiments using evaluation metrics.
5. Tune LightGBM hyperparameters.
6. Select the best LightGBM model.
7. Save the best model.
8. Evaluate the final saved model on test data.

In [23]:
import sys
from pathlib import Path

project_root = Path.cwd().resolve()

while not (project_root / "src").exists():
    if project_root == project_root.parent:
        raise RuntimeError("Could not find project root containing 'src'")
    project_root = project_root.parent

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print("Project root:", project_root)

Project root: C:\Users\user\Desktop\Kolia Stuff\SP26\Data Science\Project\Data-Science-Project


In [24]:
import lightgbm as lgb
import pandas as pd
import joblib
from src.diabetes_prediction.transformation.transformation import DataTransformation
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    matthews_corrcoef,
)

### Load Data

In [25]:
train_transformed = pd.read_csv('../../data/preprocessed/train_preprocessed.csv')
x_train_transformed = train_transformed.drop("diabetes", axis=1)
y_train_transformed = train_transformed["diabetes"]

val = pd.read_csv('../../data/preprocessed/validation_preprocessed.csv')
x_val = val.drop("diabetes", axis=1)
y_val = val["diabetes"]

print(f'x_train_transformed: {train_transformed.shape}')
train_transformed.head()

x_train_transformed: (67139, 21)


,smoking_history_current,smoking_history_ever,smoking_history_former,smoking_history_never,smoking_history_not current,age,bmi,HbA1c_level,blood_glucose_level,glucose_hba1c_interaction,...,age_bmi_interaction,bmi_hba1c_interaction,age_glucose_interaction,gender,hypertension,heart_disease,high_hba1c_flag,senior_flag,cardio_risk_flag,diabetes
0,0.0,0.0,0.0,0.0,0.0,0.612112,-0.637209,0.000000,-0.677966,-0.459103,...,-0.056612,-0.232854,-0.081827,0,0,0,0,0,0,0
1,0.0,0.0,1.0,0.0,0.0,0.737237,0.818605,0.500000,0.000000,0.411609,...,0.658534,1.018393,0.557564,0,0,0,0,0,0,0
2,0.0,0.0,1.0,0.0,0.0,0.399399,-0.229457,0.000000,0.254237,0.382586,...,-0.339001,0.014118,-0.070409,1,0,0,0,0,0,0
3,0.0,1.0,0.0,0.0,0.0,0.874875,1.485271,0.500000,0.254237,0.668865,...,1.258590,1.470922,1.050428,0,0,0,0,1,0,0
4,0.0,0.0,0.0,1.0,0.0,0.687187,-0.395349,-1.642857,1.016949,-0.142480,...,0.148131,-1.008759,1.078972,1,0,0,0,0,0,0


In [26]:
train_ADASYN = pd.read_csv('../../data/imbalance_resolve/ADASYN.csv')
x_train_ADASYN = train_ADASYN.drop("diabetes_target", axis=1)
y_train_ADASYN = train_ADASYN["diabetes_target"]
print(f'train_ADASYN: {train_ADASYN.shape}')

train_smote_enn = pd.read_csv('../../data/imbalance_resolve/train_smote_enn.csv')
x_train_smote_enn = train_smote_enn.drop("diabetes_target", axis=1)
y_train_smote_enn = train_smote_enn["diabetes_target"]
print(f'train_smote_enn: {train_smote_enn.shape}')

train_smote_tomek = pd.read_csv('../../data/imbalance_resolve/train_smote_tomek.csv')
x_train_smote_tomek = train_smote_tomek.drop("diabetes_target", axis=1)
y_train_smote_tomek = train_smote_tomek["diabetes_target"]
print(f'train_smote_tomek: {train_smote_tomek.shape}')

train_smote = pd.read_csv('../../data/imbalance_resolve/train_smote.csv')
x_train_smote = train_smote.drop("diabetes_target", axis=1)
y_train_smote = train_smote["diabetes_target"]
print(f'train_smote: {train_smote.shape}')

train_ADASYN: (122666, 16)
train_smote_enn: (112303, 16)
train_smote_tomek: (121658, 16)
train_smote: (112303, 16)


### Evaluation Metrics
- Accuracy
- **Precision**: When I say diabetic, how often am I right?, TP / (TP + FP)
- **Recall**: How many diabetics did I catch?, TP / (TP + FN) :: **most important**
- **F1-score**
- ROC-AUC
- **PR-AUC**
- **MCC**

`it is more important to us to say that a patient has diabetes when he has, and missing that would be bad that is why recall is most important here`

#### Helper functions for model evaluation

In [27]:
def evaluate_model(model, X, y, threshold=0.5):
    y_proba = model.predict_proba(X)[:, 1]

    y_pred = (y_proba >= threshold).astype(int)

    results = {
        "Accuracy": accuracy_score(y, y_pred),
        "Precision": precision_score(y, y_pred, zero_division=0),
        "Recall": recall_score(y, y_pred),
        "F1-score": f1_score(y, y_pred),
        "ROC-AUC": roc_auc_score(y, y_proba),
        "PR-AUC": average_precision_score(y, y_proba),
        "MCC": matthews_corrcoef(y, y_pred)
    }

    return results

def print_results(title, results):
    print(f"========== {title} ==========")
    for k, v in results.items():
        print(f"{k}: {v:.4f}")
    print()


def evaluate_and_store(results_table, name, model, X, y, threshold=0.5):
    results = evaluate_model(model, X, y, threshold=threshold)
    print_results(name, results)

    row = {"Experiment": name}
    row.update(results)
    results_table.append(row)

## Experiment 1 — LightGBM on Transformed Train Data

In [28]:
lgbm_results_table = []

model_lgbm_transformed = lgb.LGBMClassifier(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    verbose=-1,
    n_jobs=-1,
)

model_lgbm_transformed.fit(x_train_transformed, y_train_transformed)

evaluate_and_store(
    lgbm_results_table,
    "LightGBM - transformed",
    model_lgbm_transformed,
    x_val,
    y_val,
)

========== LightGBM - transformed ==========
Accuracy: 0.9720
Precision: 0.9780
Recall: 0.6989
F1-score: 0.8152
ROC-AUC: 0.9782
PR-AUC: 0.8894
MCC: 0.8138



## Experiment 2 — LightGBM on Transformed Train Data with Class Weights

In [29]:
model_lgbm_transformed_w = lgb.LGBMClassifier(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    class_weight="balanced",
    random_state=42,
    verbose=-1,
    n_jobs=-1,
)

model_lgbm_transformed_w.fit(x_train_transformed, y_train_transformed)

evaluate_and_store(
    lgbm_results_table,
    "LightGBM - transformed - class_weight balanced",
    model_lgbm_transformed_w,
    x_val,
    y_val,
)

========== LightGBM - transformed - class_weight balanced ==========
Accuracy: 0.9111
Precision: 0.4985
Recall: 0.9112
F1-score: 0.6444
ROC-AUC: 0.9777
PR-AUC: 0.8876
MCC: 0.6342



## Experiment 3 — LightGBM on ADASYN Train Data

In [30]:
model_lgbm_adasyn = lgb.LGBMClassifier(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    verbose=-1,
    n_jobs=-1,
)

model_lgbm_adasyn.fit(x_train_ADASYN, y_train_ADASYN)

evaluate_and_store(
    lgbm_results_table,
    "LightGBM - ADASYN",
    model_lgbm_adasyn,
    x_val[x_train_ADASYN.columns],
    y_val,
)

========== LightGBM - ADASYN ==========
Accuracy: 0.9373
Precision: 0.6043
Recall: 0.8428
F1-score: 0.7039
ROC-AUC: 0.9760
PR-AUC: 0.8813
MCC: 0.6815



## Experiment 4 — LightGBM on SMOTE Train Data

In [31]:
model_lgbm_smote = lgb.LGBMClassifier(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    verbose=-1,
    n_jobs=-1,
)

model_lgbm_smote.fit(x_train_smote, y_train_smote)

evaluate_and_store(
    lgbm_results_table,
    "LightGBM - SMOTE",
    model_lgbm_smote,
    x_val[x_train_smote.columns],
    y_val,
)

========== LightGBM - SMOTE ==========
Accuracy: 0.9323
Precision: 0.5774
Recall: 0.8742
F1-score: 0.6954
ROC-AUC: 0.9776
PR-AUC: 0.8879
MCC: 0.6771



## Experiment 5 — LightGBM on SMOTE-ENN Train Data

In [32]:
model_lgbm_smote_enn = lgb.LGBMClassifier(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    verbose=-1,
    n_jobs=-1,
)

model_lgbm_smote_enn.fit(x_train_smote_enn, y_train_smote_enn)

evaluate_and_store(
    lgbm_results_table,
    "LightGBM - SMOTE-ENN",
    model_lgbm_smote_enn,
    x_val[x_train_smote_enn.columns],
    y_val,
)

========== LightGBM - SMOTE-ENN ==========
Accuracy: 0.9323
Precision: 0.5774
Recall: 0.8742
F1-score: 0.6954
ROC-AUC: 0.9776
PR-AUC: 0.8879
MCC: 0.6771



## Experiment 6 — LightGBM on SMOTE-TOMEK Train Data

In [33]:
model_lgbm_smote_tomek = lgb.LGBMClassifier(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    verbose=-1,
    n_jobs=-1,
)

model_lgbm_smote_tomek.fit(x_train_smote_tomek, y_train_smote_tomek)

evaluate_and_store(
    lgbm_results_table,
    "LightGBM - SMOTE-TOMEK",
    model_lgbm_smote_tomek,
    x_val[x_train_smote_tomek.columns],
    y_val,
)

========== LightGBM - SMOTE-TOMEK ==========
Accuracy: 0.9615
Precision: 0.7813
Recall: 0.7838
F1-score: 0.7826
ROC-AUC: 0.9776
PR-AUC: 0.8877
MCC: 0.7615



In [34]:
df = pd.DataFrame(lgbm_results_table)
df.head(10)

,Experiment,Accuracy,Precision,Recall,F1-score,ROC-AUC,PR-AUC,MCC
0,LightGBM - transformed,0.971989,0.977998,0.698899,0.815222,0.978245,0.889353,0.813765
1,LightGBM - transformed - class_weight balanced,0.911100,0.498495,0.911164,0.644426,0.977720,0.887611,0.634183
2,LightGBM - ADASYN,0.937305,0.604284,0.842767,0.703874,0.975954,0.881326,0.681477
3,LightGBM - SMOTE,0.932300,0.577362,0.874214,0.695435,0.977586,0.887899,0.677108
4,LightGBM - SMOTE-ENN,0.932300,0.577362,0.874214,0.695435,0.977586,0.887899,0.677108
5,LightGBM - SMOTE-TOMEK,0.961493,0.781348,0.783805,0.782575,0.977641,0.887708,0.761452


## LightGBM Experiments Evaluation Results

The table below compares the performance of different LightGBM experiments on the validation set.

| Experiment | Accuracy | Precision | Recall | F1-score | ROC-AUC | PR-AUC | MCC |
|---|---:|---:|---:|---:|---:|---:|---:|
| LightGBM - transformed | `0.9720` | `0.9780` | 0.6989 | `0.8152` | 0.9782 | 0.8894 | `0.8138` |
| LightGBM - transformed - class_weight balanced | 0.9111 | 0.4985 | 0.9112 | 0.6444 | 0.9777 | 0.8876 | 0.6342 |
| LightGBM - ADASYN | 0.9373 | 0.6043 | 0.8428 | 0.7039 | 0.9760 | 0.8813 | 0.6815 |
| LightGBM - SMOTE | 0.9323 | 0.5774 | `0.8742` | 0.6954 | 0.9776 | `0.8879` | 0.6771 |
| LightGBM - SMOTE-ENN | 0.9323 | 0.5774 | `0.8742` | 0.6954 | `0.9776` | `0.8879` | 0.6771 |
| LightGBM - SMOTE-TOMEK | 0.9615 | 0.7813 | 0.7838 | 0.7826 | 0.9776 | 0.8877 | 0.7615 |

#### I will continue with ADASYN to tune the hyperparameters.

ADASYN achieves a strong **Recall** (0.8428) with the best **Precision** among resampled datasets (0.6043) and the highest **MCC** (0.6815), making it the most balanced starting point for tuning. SMOTE and SMOTE-ENN produce identical results with slightly higher recall but weaker precision and MCC.

## Tuning Hyperparameters using x_train_ADASYN

| Hyperparameter | Meaning | Importance |
|---|---|---|
| `n_estimators` | Number of boosting rounds (trees). | More trees can improve performance but risk overfitting. |
| `learning_rate` | Step size for each boosting round. | Smaller values require more trees but improve generalization. |
| `max_depth` | Maximum depth of each tree. | Controls model complexity; deeper trees can capture more patterns. |
| `num_leaves` | Maximum number of leaves in one tree. | LightGBM grows leaf-wise; more leaves = more complex model. |
| `subsample` | Fraction of training samples used per tree. | Reduces overfitting and speeds up training. |
| `colsample_bytree` | Fraction of features used per tree. | Adds randomness to reduce overfitting. |
| `reg_alpha` | L1 regularization term. | Encourages sparsity and reduces overfitting. |
| `reg_lambda` | L2 regularization term. | Penalizes large weights to improve generalization. |
| `min_child_samples` | Minimum samples required in a leaf. | Prevents overfitting on small groups. |

#### Experiment 1
Baseline — `n_estimators=300`, `learning_rate=0.05`, `max_depth=6`

In [35]:
lgbm_tuning_results = []

model1 = lgb.LGBMClassifier(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    verbose=-1,
    n_jobs=-1,
)

model1.fit(x_train_smote_tomek, y_train_smote_tomek)

evaluate_and_store(
    lgbm_tuning_results,
    "LightGBM - E1 Tuning | n_estimators=300 | lr=0.05 | max_depth=6",
    model1,
    x_val[x_train_smote_tomek.columns],
    y_val,
)

========== LightGBM - E1 Tuning | n_estimators=300 | lr=0.05 | max_depth=6 ==========
Accuracy: 0.9615
Precision: 0.7813
Recall: 0.7838
F1-score: 0.7826
ROC-AUC: 0.9776
PR-AUC: 0.8877
MCC: 0.7615



#### Experiment 2
Increase `n_estimators` to allow more boosting rounds

In [36]:
model2 = lgb.LGBMClassifier(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    verbose=-1,
    n_jobs=-1,
)

model2.fit(x_train_smote_tomek, y_train_smote_tomek)

evaluate_and_store(
    lgbm_tuning_results,
    "LightGBM - E2 Tuning | n_estimators=500 | lr=0.05 | max_depth=6",
    model2,
    x_val[x_train_smote_tomek.columns],
    y_val,
)

========== LightGBM - E2 Tuning | n_estimators=500 | lr=0.05 | max_depth=6 ==========
Accuracy: 0.9634
Precision: 0.8028
Recall: 0.7775
F1-score: 0.7899
ROC-AUC: 0.9780
PR-AUC: 0.8889
MCC: 0.7700



#### Experiment 3
Lower `learning_rate` to `0.01` — slower but potentially better generalization

In [37]:
model3 = lgb.LGBMClassifier(
    n_estimators=500,
    learning_rate=0.01,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    verbose=-1,
    n_jobs=-1,
)

model3.fit(x_train_smote_tomek, y_train_smote_tomek)

evaluate_and_store(
    lgbm_tuning_results,
    "LightGBM - E3 Tuning | n_estimators=500 | lr=0.01 | max_depth=6",
    model3,
    x_val[x_train_smote_tomek.columns],
    y_val,
)

========== LightGBM - E3 Tuning | n_estimators=500 | lr=0.01 | max_depth=6 ==========
Accuracy: 0.9438
Precision: 0.6404
Recall: 0.8318
F1-score: 0.7237
ROC-AUC: 0.9752
PR-AUC: 0.8794
MCC: 0.7003



#### Experiment 4
Increase `max_depth` to `8` to allow deeper trees

In [38]:
model4 = lgb.LGBMClassifier(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=8,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    verbose=-1,
    n_jobs=-1,
)

model4.fit(x_train_smote_tomek, y_train_smote_tomek)

evaluate_and_store(
    lgbm_tuning_results,
    "LightGBM - E4 Tuning | n_estimators=300 | lr=0.05 | max_depth=8",
    model4,
    x_val[x_train_smote_tomek.columns],
    y_val,
)

========== LightGBM - E4 Tuning | n_estimators=300 | lr=0.05 | max_depth=8 ==========
Accuracy: 0.9639
Precision: 0.8094
Recall: 0.7744
F1-score: 0.7915
ROC-AUC: 0.9783
PR-AUC: 0.8899
MCC: 0.7720



#### Experiment 5
Add L1/L2 regularization and increase `min_child_samples` to reduce overfitting

In [39]:
model5 = lgb.LGBMClassifier(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=1.0,
    min_child_samples=20,
    random_state=42,
    verbose=-1,
    n_jobs=-1,
)

model5.fit(x_train_smote_tomek, y_train_smote_tomek)

evaluate_and_store(
    lgbm_tuning_results,
    "LightGBM - E5 Tuning | n_estimators=300 | lr=0.05 | reg_alpha=0.1 | reg_lambda=1.0",
    model5,
    x_val[x_train_smote_tomek.columns],
    y_val,
)

========== LightGBM - E5 Tuning | n_estimators=300 | lr=0.05 | reg_alpha=0.1 | reg_lambda=1.0 ==========
Accuracy: 0.9612
Precision: 0.7776
Recall: 0.7862
F1-score: 0.7819
ROC-AUC: 0.9775
PR-AUC: 0.8870
MCC: 0.7606



#### Experiment 6
Combine deeper trees, more estimators, and regularization with increased `num_leaves`

In [40]:
model6 = lgb.LGBMClassifier(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=8,
    num_leaves=63,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=1.0,
    min_child_samples=20,
    random_state=42,
    verbose=-1,
    n_jobs=-1,
)

model6.fit(x_train_smote_tomek, y_train_smote_tomek)

evaluate_and_store(
    lgbm_tuning_results,
    "LightGBM - E6 Tuning | n_estimators=500 | lr=0.05 | max_depth=8 | num_leaves=63",
    model6,
    x_val[x_train_smote_tomek.columns],
    y_val,
)

========== LightGBM - E6 Tuning | n_estimators=500 | lr=0.05 | max_depth=8 | num_leaves=63 ==========
Accuracy: 0.9682
Precision: 0.8497
Recall: 0.7775
F1-score: 0.8120
ROC-AUC: 0.9797
PR-AUC: 0.8982
MCC: 0.7956



In [41]:
df_tuning = pd.DataFrame(lgbm_tuning_results)
df_tuning

,Experiment,Accuracy,Precision,Recall,F1-score,ROC-AUC,PR-AUC,MCC
0,LightGBM - E1 Tuning | n_estimators=300 | lr=0...,0.961493,0.781348,0.783805,0.782575,0.977641,0.887708,0.761452
1,LightGBM - E2 Tuning | n_estimators=500 | lr=0...,0.963439,0.802760,0.777516,0.789936,0.977952,0.888870,0.770037
2,LightGBM - E3 Tuning | n_estimators=500 | lr=0...,0.943838,0.640436,0.831761,0.723666,0.975209,0.879403,0.700334
3,LightGBM - E4 Tuning | n_estimators=300 | lr=0...,0.963926,0.809367,0.774371,0.791483,0.978333,0.889934,0.771974
4,LightGBM - E5 Tuning | n_estimators=300 | lr=0...,0.961215,0.777605,0.786164,0.781861,0.977525,0.886951,0.760591
5,LightGBM - E6 Tuning | n_estimators=500 | lr=0...,0.968166,0.849656,0.777516,0.811987,0.979683,0.898173,0.795568


## LightGBM Hyperparameter Tuning Results

| Experiment | Accuracy | Precision | Recall | F1-score | ROC-AUC | PR-AUC | MCC |
|---|---:|---:|---:|---:|---:|---:|---:|
| LightGBM - E1 Tuning \| n_estimators=300 \| lr=0.05 \| max_depth=6 | 0.9373 | 0.6043 | 0.8428 | 0.7039 | 0.9760 | 0.8813 | 0.6815 |
| LightGBM - E2 Tuning \| n_estimators=500 \| lr=0.05 \| max_depth=6 | 0.9493 | 0.6794 | 0.8082 | 0.7382 | 0.9755 | 0.8787 | 0.7137 |
| LightGBM - E3 Tuning \| n_estimators=500 \| lr=0.01 \| max_depth=6 | 0.8912 | 0.4440 | `0.9167` | 0.5983 | 0.9745 | 0.8744 | 0.5919 |
| LightGBM - E4 Tuning \| n_estimators=300 \| lr=0.05 \| max_depth=8 | 0.9468 | 0.6583 | 0.8270 | 0.7331 | 0.9761 | 0.8807 | 0.7096 |
| LightGBM - E5 Tuning \| n_estimators=300 \| lr=0.05 \| reg_alpha=0.1 \| reg_lambda=1.0 | 0.9430 | 0.6347 | 0.8373 | 0.7220 | `0.9767` | `0.8844` | 0.6992 |
| LightGBM - E6 Tuning \| n_estimators=500 \| lr=0.05 \| max_depth=8 \| num_leaves=63 | `0.9583` | `0.7489` | 0.7948 | `0.7712` | `0.9767` | `0.8864` | `0.7486` |

### Interpretation

The tuning results reveal a clear trade-off between **Recall** and overall predictive quality.

**E3** (low learning rate) achieved the highest **Recall** (0.9167) by learning more slowly and producing softer decision boundaries, but at the cost of much lower **Precision** (0.4440), **F1-score** (0.5983), and **MCC** (0.5919).

**E6** (deeper trees, more estimators, regularization, increased `num_leaves`) achieved the best **Accuracy** (0.9583), **Precision** (0.7489), **F1-score** (0.7712), **ROC-AUC** (0.9767), **PR-AUC** (0.8864), and **MCC** (0.7486), offering the strongest overall model.

Since **Recall is the top priority** in this project — missing a diabetic patient is more harmful than a false alarm — **E3** is selected as the best model.

### Selected Model

Based on the validation results, the best LightGBM model is:

> **LightGBM - E3 Tuning | n_estimators=500 | lr=0.01 | max_depth=6**

This model achieves the highest **Recall** (0.9167), ensuring the maximum detection of diabetic patients, which is the primary objective of this project.

### Best Model Results

| Metric | Value |
|---|---:|
| Accuracy | 0.8912 |
| Precision | 0.4440 |
| Recall | 0.9167 |
| F1-score | 0.5983 |
| ROC-AUC | 0.9745 |
| PR-AUC | 0.8744 |
| MCC | 0.5919 |

#### Parameters for the Selected LightGBM Model

| Parameter | Value | Description |
|---|---|---|
| n_estimators | 500 | Number of boosting rounds |
| learning_rate | 0.01 | Smaller step size for better generalization |
| max_depth | 6 | Maximum depth of each tree |
| subsample | 0.8 | Fraction of training samples used per tree |
| colsample_bytree | 0.8 | Fraction of features used per tree |
| random_state | 42 | Seed for reproducibility |

## Final Evaluation On Test Data

In [42]:
transformer = DataTransformation()
transformer.load_preprocessor()
test = pd.read_csv('../../data/split/test.csv')
x_test = test.drop("diabetes", axis=1)
y_test = test["diabetes"]
x_test_transformed = transformer.transform(x_test)
x_test_transformed = x_test_transformed[x_train_smote_tomek.columns]

c:\Users\user\anaconda3\Lib\site-packages\sklearn\base.py:380: InconsistentVersionWarning: Trying to unpickle estimator OneHotEncoder from version 1.8.0 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
c:\Users\user\anaconda3\Lib\site-packages\sklearn\base.py:380: InconsistentVersionWarning: Trying to unpickle estimator MinMaxScaler from version 1.8.0 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
c:\Users\user\anaconda3\Lib\site-packages\sklearn\base.py:380: InconsistentVersionWarning: Trying to unpickle estimator RobustScaler from version 1.8.0 when using version 1.6.1. This might lead to breaking code or

In [46]:
lgbm_final_result = []

evaluate_and_store(
    lgbm_final_result,
    "LightGBM - SMOTE TOMEK Final",
    model2,  # best tuned model: n_estimators=500, lr=0.01, max_depth=6
    x_test_transformed,
    y_test,
)

========== LightGBM - SMOTE TOMEK Final ==========
Accuracy: 0.9590
Precision: 0.7706
Recall: 0.7634
F1-score: 0.7670
ROC-AUC: 0.9766
PR-AUC: 0.8771
MCC: 0.7445



In [44]:
joblib.dump(model3, "../../models/LightGBM_model.pkl")  # best tuned model: n_estimators=500, lr=0.01, max_depth=6

['../../models/LightGBM_model.pkl']